In [4]:
"""LoRA-Fine-Tuning von Qwen3-0.6B für Transaktionskategorisierung.

Erwartet transactions.csv mit Spalten: text, label
(text = bereinigter Empfänger + Verwendungszweck, label = Unterkategorie)
"""

import pandas as pd
from datasets import Dataset
from peft import LoraConfig
from sklearn.model_selection import train_test_split
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen3-0.6B"

In [5]:
# ---------- Daten laden ----------
df = pd.read_csv("dkb_synth_large.csv", sep=";", encoding="utf-8", skiprows=5)
df["text"] = df["Zahlungsempfänger*in"].fillna("") + " " + df["Verwendungszweck"].fillna("")
df["label"] = df["Unterkategorie"]
df = df.dropna(subset=["label"])

labels = sorted(df["label"].unique())
label_list = ", ".join(labels)

PROMPT = (
    "Ordne die folgende Banktransaktion genau einer Kategorie zu. "
    f"Antworte nur mit dem Kategorienamen.\nKategorien: {label_list}\n"
    "Transaktion: {text}"
)

def to_chat(row):
    return {
        "messages": [
            {"role": "user", "content": PROMPT.format(text=row["text"])},
            {"role": "assistant", "content": row["label"]},
        ]
    }

train_df, val_df = train_test_split(
    df, test_size=0.15, stratify=df["label"], random_state=42
)
train_ds = Dataset.from_list([to_chat(r) for _, r in train_df.iterrows()])
val_ds = Dataset.from_list([to_chat(r) for _, r in val_df.iterrows()])

In [6]:
# ---------- LoRA-Konfiguration ----------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# ---------- Training ----------
args = SFTConfig(
    output_dir="checkpoints-llm",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    max_length=512,
    completion_only_loss=True,  # Loss nur auf der Assistant-Antwort (dem Label)
)

trainer = SFTTrainer(
    model=MODEL_NAME,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
)

trainer.train()

# LoRA-Adapter + Tokenizer speichern
trainer.save_model("txn-llm-classifier")

Building labels for eval dataset: 100%|██████████| 396/396 [00:00<00:00, 11398.66 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/clarabrilke/ML projects/moneta/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# ---------- Evaluation: exakte Label-Trefferquote ----------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto")
model = PeftModel.from_pretrained(base, "txn-llm-classifier").eval()

correct = 0
for _, row in val_df.iterrows():
    msgs = [{"role": "user", "content": PROMPT.format(text=row["text"])}]
    inputs = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt",
        enable_thinking=False,
    )
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=10, do_sample=False)
    pred = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    correct += int(pred == row["label"])

print(f"Exact-Match Accuracy: {correct / len(val_df):.3f}")